# 04 Inductive models: similarity kNN, two-tower network, ensemble (C-T and H-T)

These models score a compound-target pair from the compound's chemistry and the target's sequence, so they also work for compounds
and targets with no known links.
- **Similarity kNN**: Tanimoto similarity on ECFP4. score(c, t) = summed similarity of the k most similar training compounds that hit t.
  k is picked on validation MRR. It cannot score a target with no training links.
- **Two-tower (ESM-2)**: compound tower (ECFP4 counts, FCFP4, MACCS, herb membership) and target tower (ESM-2), scaled cosine similarity
  plus a per-target bias, softmax over training targets with the compound's other training targets masked.
- **Two-tower (AAC)**: the same with AAC instead of ESM-2, to show what ESM-2 adds.
- **Ensemble**: weighted average of the per-compound ranks of two-tower (ESM-2) and kNN. The weight (0, 0.25, 0.5, 0.75 or 1) is picked on
  validation MRR; 1 means two-tower only and 0 means kNN only.

Reads: `prep/` &nbsp; Writes: `results/metrics_inductive.csv`, `results/curves_inductive.csv`

## Setup

In [1]:
import hti
import time
import pandas as pd
from hti.data import Prep, RESULTS
from hti.metrics import evaluate, val_mrr
from hti.models import knn_scores, row_ranks, tanimoto_matrix
from hti.train import train_two_tower

RESULTS.mkdir(exist_ok=True)
prep = Prep()
CT_TASKS = ['ct_edge', 'ct_compound', 'ct_target']
similarity = tanimoto_matrix(prep.tensor(prep.ecfp4))
print(prep.device, similarity.shape)

cuda torch.Size([2319, 2319])


## All C-T tasks and folds
About 30 two-tower models plus kNN; roughly 10 to 20 minutes on an RTX 4060.

In [2]:
records, curves = [], []
start = time.time()

def record(task, k, name, scores, fold, with_loss, info=None):
    result = evaluate(scores, fold, prep, 'ct', with_loss=with_loss)
    for part, m in result.items():
        records.append({'task': task, 'model': name, 'fold': k, 'set': part, **(info or {}), **m})
    test = result['test']
    ht = f"  HT MRR {test['HT MRR']:.3f}" if 'HT MRR' in test else ''
    print(f"{task:12s} {name:18s} fold {k}  test MRR {test['MRR']:.3f}  Hits@10 {test['Hits@10']:.3f}{ht}  {time.time() - start:5.0f}s")

for task in CT_TASKS:
    for k in range(prep.k):
        fold = prep.fold(task, k)
        train_pos = fold['train']['pos']

        best_k = max([5, 10, 25, 50], key=lambda q: val_mrr(
            knn_scores(similarity, train_pos, prep.n_compounds, prep.n_targets, q), fold['val'], prep.known['ct']))
        knn = knn_scores(similarity, train_pos, prep.n_compounds, prep.n_targets, best_k)
        record(task, k, 'kNN', knn, fold, False, {'k': best_k})

        for name, feature in [('Two-tower (ESM-2)', 'esm'), ('Two-tower (AAC)', 'aac')]:
            scores, info, curve = train_two_tower(prep, fold, target_feature=feature, seed=k)
            record(task, k, name, scores, fold, True, info)
            curves += [{'task': task, 'model': name, 'fold': k, **c} for c in curve]
            if feature == 'esm':
                two_tower = scores

        # weight 1 means two-tower only, 0 means kNN only, so the ensemble can never do worse on validation than either one
        a, b = row_ranks(two_tower), row_ranks(knn)
        w = max([0.0, 0.25, 0.5, 0.75, 1.0], key=lambda x: val_mrr(x * a + (1 - x) * b, fold['val'], prep.known['ct']))
        record(task, k, 'Ensemble', w * a + (1 - w) * b, fold, False, {'weight': w})

metrics = pd.DataFrame(records)
metrics.to_csv(RESULTS / 'metrics_inductive.csv', index=False)
pd.DataFrame(curves).to_csv(RESULTS / 'curves_inductive.csv', index=False)

ct_edge      kNN                fold 0  test MRR 0.103  Hits@10 0.209  HT MRR 0.083      2s
ct_edge      Two-tower (ESM-2)  fold 0  test MRR 0.362  Hits@10 0.519  HT MRR 0.083     12s
ct_edge      Two-tower (AAC)    fold 0  test MRR 0.299  Hits@10 0.445  HT MRR 0.083     26s
ct_edge      Ensemble           fold 0  test MRR 0.362  Hits@10 0.519  HT MRR 0.079     27s
ct_edge      kNN                fold 1  test MRR 0.119  Hits@10 0.225  HT MRR 0.079     29s
ct_edge      Two-tower (ESM-2)  fold 1  test MRR 0.337  Hits@10 0.511  HT MRR 0.083     36s
ct_edge      Two-tower (AAC)    fold 1  test MRR 0.269  Hits@10 0.412  HT MRR 0.081     48s
ct_edge      Ensemble           fold 1  test MRR 0.337  Hits@10 0.511  HT MRR 0.080     49s
ct_edge      kNN                fold 2  test MRR 0.098  Hits@10 0.199  HT MRR 0.063     50s
ct_edge      Two-tower (ESM-2)  fold 2  test MRR 0.343  Hits@10 0.511  HT MRR 0.067     57s
ct_edge      Two-tower (AAC)    fold 2  test MRR 0.292  Hits@10 0.440  HT MRR 0.

## Mean and std over the 5 folds, for train, val and test
kNN and the ensemble are not trained with a loss, so their loss is empty.

In [3]:
cols = ['PR-AUC', 'Hits@10', 'Hits@50', 'MRR', 'loss', 'HT PR-AUC', 'HT Hits@10', 'HT Hits@50', 'HT MRR']
g = metrics.groupby(['task', 'model', 'set'])[cols]
table = (g.mean().round(3).astype(str) + ' ± ' + g.std().round(3).astype(str)).replace('nan ± nan', '')
table.reindex(CT_TASKS, level=0)

PR-AUC        Hits@10  \
task        model             set                                   
ct_edge     Ensemble          test   0.159 ± 0.004   0.51 ± 0.012   
                              train  0.789 ± 0.011  0.881 ± 0.034   
                              val    0.467 ± 0.013  0.512 ± 0.008   
            Two-tower (AAC)   test   0.216 ± 0.016  0.426 ± 0.016   
                              train  0.816 ± 0.009  0.779 ± 0.037   
                              val    0.506 ± 0.024  0.419 ± 0.011   
            Two-tower (ESM-2) test   0.238 ± 0.015   0.51 ± 0.012   
                              train  0.874 ± 0.014  0.881 ± 0.034   
                              val    0.552 ± 0.023  0.512 ± 0.008   
            kNN               test   0.186 ± 0.009  0.208 ± 0.011   
                              train  0.895 ± 0.039  0.781 ± 0.091   
                              val    0.478 ± 0.013  0.215 ± 0.011   
ct_compound Ensemble          test    0.128 ± 0.02  0.229 ± 0.045   
                              train  0.676 ± 0.014  0.725 ± 0.037   
                              val    0.416 ± 0.034  0.262 ± 0.047   
            Two-tower (AAC)   test   0.093 ± 0.025  0.212 ± 0.033   
                              train  0.643 ± 0.086  0.655 ± 0.036   
                              val    0.343 ± 0.046  0.245 ± 0.038   
            Two-tower (ESM-2) test    0.13 ± 0.028  0.229 ± 0.045   
                              train  0.755 ± 0.023  0.725 ± 0.037   
                              val     0.418 ± 0.05  0.262 ± 0.047   
            kNN               test   0.188 ± 0.033   0.217 ± 0.04   
                              train  0.904 ± 0.055  0.853 ± 0.103   
                              val    0.476 ± 0.053   0.24 ± 0.042   
ct_target   Ensemble          test    0.115 ± 0.01  0.375 ± 0.033   
                              train  0.718 ± 0.039  0.839 ± 0.089   
                              val    0.352 ± 0.017  0.459 ± 0.029   
            Two-tower (AAC)   test    0.08 ± 0.025  0.248 ± 0.027   
                              train  0.606 ± 0.152   0.61 ± 0.149   
                              val    0.244 ± 0.082  0.309 ± 0.028   
            Two-tower (ESM-2) test   0.192 ± 0.029  0.375 ± 0.033   
                              train  0.819 ± 0.055  0.839 ± 0.089   
                              val    0.415 ± 0.026  0.459 ± 0.029   
            kNN               test      0.02 ± 0.0      0.0 ± 0.0   
                              train  0.936 ± 0.005  0.939 ± 0.008   
                              val      0.091 ± 0.0      0.0 ± 0.0   

                                           Hits@50            MRR  \
task        model             set                                   
ct_edge     Ensemble          test   0.729 ± 0.003  0.342 ± 0.015   
                              train  0.961 ± 0.013   0.77 ± 0.045   
                              val     0.73 ± 0.009  0.342 ± 0.003   
            Two-tower (AAC)   test   0.621 ± 0.014  0.285 ± 0.012   
                              train  0.885 ± 0.022  0.668 ± 0.047   
                              val    0.616 ± 0.018  0.285 ± 0.007   
            Two-tower (ESM-2) test   0.729 ± 0.003  0.342 ± 0.015   
                              train  0.961 ± 0.013   0.77 ± 0.045   
                              val     0.73 ± 0.009  0.342 ± 0.003   
            kNN               test   0.462 ± 0.011  0.107 ± 0.008   
                              train   0.97 ± 0.023  0.506 ± 0.089   
                              val    0.475 ± 0.009  0.108 ± 0.006   
ct_compound Ensemble          test    0.439 ± 0.06   0.14 ± 0.022   
                              train  0.863 ± 0.025   0.61 ± 0.043   
                              val    0.477 ± 0.045  0.178 ± 0.037   
            Two-tower (AAC)   test   0.383 ± 0.045  0.134 ± 0.015   
                              train  0.792 ± 0.026  0.549 ± 0.034   
                              val    0.414 ± 0.038  0.172 ± 0.033   
            Two-tower (ESM-2) test    0.4